In [7]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [8]:
from optcutfreq import optcutfreq
import numpy as np
import pandas as pd
import scipy, os, pickle
from scipy import signal
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

**Read**

In [9]:
#Function
def Get_Vertical_Acceleration(Dataset):
        ''' Function to determine vertical axis (i.e. column with mean acc closest to g) and randomly assign the two other columns as ML or AP

            Input: 
                - Dataset: DF of 4 columns like (timestamp, x, y, z)
            Output: 
                - COLUMNS_OF_INTEREST : List of 4 element like (timestamp, vertical, randomML, randomAP)
        '''
        g = 9.81
        mean_Column_1 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 1].values), np.where(np.abs(Dataset.iloc[:, 1].values) * 0 !=0)[0])))
        mean_Column_2 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 2].values), np.where(np.abs(Dataset.iloc[:, 2].values) * 0 !=0)[0])))
        mean_Column_3 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 3].values), np.where(np.abs(Dataset.iloc[:, 3].values) * 0 !=0)[0])))
        array = np.array([mean_Column_1, mean_Column_2, mean_Column_3])
        if np.argmin(array) == 0:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[1]
            ML_ACC_COLUMN_LABEL = Dataset.columns[2] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[3] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]

        if np.argmin(array) == 1:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[2]
            ML_ACC_COLUMN_LABEL = Dataset.columns[1] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[3] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]

        if np.argmin(array) == 2:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[3]
            ML_ACC_COLUMN_LABEL = Dataset.columns[1] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[2] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]
        
        return COLUMNS_OF_INTEREST

**Store**

In [10]:
# Declare
    # Signal 
        # HOME SIGNAL
HOME_EXTRACTED_SIGNAL_PATH = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/HomeDay1/"
HOME_EXTRACTED_DATA_FILES_LIST = np.sort(np.array(os.listdir(HOME_EXTRACTED_SIGNAL_PATH))) # e.g PD04.csv as for events so no need to re-declare
        # SEQUENCES
SEQUENCES_PATH = "/Users/cyrilleetude/Documents/GitHub/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/2_Transform/0_Detect_Gait_Sequences/Day1/"
SEQUENCES_FILES_LIST = np.sort(np.array(os.listdir(SEQUENCES_PATH)))
IDs_HOME = [filename[:6] for filename in SEQUENCES_FILES_LIST if not filename.startswith(('.', '~'))] # Participant ID's (drop the .csv)
IDs_HOME = [filename[:-1] if filename.endswith(('.')) else filename for filename in IDs_HOME]
SEQUENCES = []

# Read 
for sequence_file in SEQUENCES_FILES_LIST:# Get files names
        df = pd.read_csv(os.path.join(SEQUENCES_PATH, sequence_file), usecols=[0, 1], engine="c")
        df = df.loc[df["End"] - df["Start"] >= 30].reset_index(drop=True) # at least 30 seconds
        SEQUENCES.append(df) # Store sequence dfs

EXTRACTED_DATA = {}
for participant in range(len(IDs_HOME)):
        # Crop Vertically to only keep time and vertical acceleration
        signal_df = pd.read_csv(f"{HOME_EXTRACTED_SIGNAL_PATH + IDs_HOME[participant]}.csv", engine="c")
        COLUMNS_OF_INTEREST_HOME = Get_Vertical_Acceleration(signal_df)  # Determine Vertical
        signal_df = signal_df[COLUMNS_OF_INTEREST_HOME[:2]]#Only keep time and vertical
        signal_df.columns = ["TIME_S", "ACC_VERTICAL_MS2"]#rename correctly
        signal_df["TIME_S"]= signal_df["TIME_S"].values - signal_df["TIME_S"].values[0]#reset time to match with detected bouts

        # Crop horizontally into sequences
        sequence_df = SEQUENCES[participant] #get sequences for a given participant in ID home
        participant_sequences = [] # Store begging and end of each sequence in lists
        for sequence in range(len(sequence_df)):
                participant_sequences.append([sequence_df.iloc[sequence,0], sequence_df.iloc[sequence,1]])
        
        bouts_of_signal_list = []
        for sequence_bout in range(len(participant_sequences)):#append bouts of signals in a list (i.e. bouts_of_signal_list is a list of dfs)
                bouts_of_signal_list.append(signal_df.loc[(signal_df["TIME_S"] >= participant_sequences[sequence_bout][0]) & (signal_df["TIME_S"] <= participant_sequences[sequence_bout][1])])

        #Reset the time of each df to 0
        for df in bouts_of_signal_list:
                df["TIME_S"] = df["TIME_S"].values - df["TIME_S"].values[0]
        
        # store every thing in a dictionarry with participant id as key and list of dfs (bouts of signals) as value
        EXTRACTED_DATA[IDs_HOME[participant]] = bouts_of_signal_list #a lits of bout of dfs


/var/folders/rm/fz861vwx62vcc5c49ww298480000gq/T/ipykernel_34803/3473093923.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TIME_S"] = df["TIME_S"].values - df["TIME_S"].values[0]
/var/folders/rm/fz861vwx62vcc5c49ww298480000gq/T/ipykernel_34803/3473093923.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TIME_S"] = df["TIME_S"].values - df["TIME_S"].values[0]
/var/folders/rm/fz861vwx62vcc5c49ww298480000gq/T/ipykernel_34803/3473093923.py:40: SettingWithCopyWarning: 
A value is trying to be se

**Save**

In [13]:
# # Define the base directory where folders will be created
# base_dir = '/Users/cyrilleetude/Documents/GitHub/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/2_Transform/1_Signal_Processing/Raw_csv_HomeDay1'

# # Create folders and save DataFrames
# for participant, raw_signal_df in EXTRACTED_DATA.items():
#     # Create the folder for the current key
#     folder_path = os.path.join(base_dir, participant)
#     os.makedirs(folder_path, exist_ok=True)
    
#     # Iterate over the list of DataFrames for the current key
#     for idx, df in enumerate(raw_signal_df):
#         # Define the filename for the DataFrame
#         file_path = os.path.join(folder_path, f'raw_signal_df_{idx + 1}.csv')
        
#         # Save the DataFrame to CSV
#         df.to_csv(file_path, index=False)

with open("/Users/cyrilleetude/Documents/GitHub/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/2_Transform/1_Signal_Processing/Raw_csv_HomeDay1/EXTRACTED_DATA.bin", "wb") as f:
            pickle.dump(EXTRACTED_DATA, f)

